# Evaluasi Ketahanan VQA Medis Berbasis BiomedCLIP terhadap Serangan Adversarial (VQA-RAD)

Notebook tunggal untuk PRD "Evaluasi Ketahanan Model VQA Medis Berbasis BiomedCLIP" (15 November 2026).
Satu notebook, satu berkas konfigurasi, seed tetap (K3, NFR-03).

> **PERINGATAN: ini bukan alat diagnosis.** Notebook dan demo hanya untuk penelitian dan pembelajaran, tidak untuk keputusan klinis (NFR-07).

**Cara memakai.** Jalankan sel dari atas ke bawah. Tiap tahap berakhir dengan sel *Checkpoint* (CP0 sampai CP6). Bila sel checkpoint gagal, perbaiki dulu sebelum lanjut (PRD bagian 9.1).
Sesi Colab/Kaggle boleh terputus: pelatihan dan serangan **dilanjutkan otomatis** dari checkpoint dan berkas hasil (R-10). Cukup jalankan ulang sel dari atas.

| Bagian | Isi | PRD |
| --- | --- | --- |
| 0 | Instalasi, konfigurasi tunggal, lingkungan | R-01, R-12, NFR-03 |
| 1 | Data VQA-RAD, split, distribusi kelas (CP0) | FR-01..04 |
| 2 | BiomedCLIP, Model 1, pelatihan 3 seed (CP1) | FR-05 |
| 3 | Pustaka serangan, evaluator, smoke test | FR-08, FR-09, NFR-02 |
| 4 | Perturbasi teks | FR-10, FR-11 |
| 5 | Skenario 1 sampai 4 pada Model 1 (CP2, CP3) | G1..G3 |
| 6 | Model 2 (TRADES), skenario 5, gradient masking (CP4) | FR-06, FR-12 |
| 7 | Tabel dan gambar hasil (CP5) | FR-14..16 |
| 8 | Demo Gradio (CP6) | FR-17, FR-18 |

**Keputusan desain (default, ubah di sel konfigurasi bila perlu).**
- Pertanyaan tertutup dievaluasi sebagai klasifikasi 2 arah (logit "no" dan "yes" dari kepala 1 kosakata jawaban). Serangan bekerja pada 2 logit itu.
- Pertanyaan terbuka: exact match pada kosakata penuh, dilaporkan terpisah dan hanya pada data bersih.
- FGSM, PGD, AutoAttack, Square memakai paket `torchattacks` (pertanyaan terbuka no. 7 PRD). Paket ini harus ditambahkan ke daftar pustaka.
- Model kontrol (encoder dibuka tanpa TRADES, pertanyaan terbuka no. 2) tersedia lewat `CFG['run_control']`, default mati.
- TRADES diimplementasikan sendiri mengikuti `trades_loss()` dari yaodongyu/TRADES (MIT), karena repositorinya bukan paket pip.

In [ ]:
# Jalankan sekali. Bila Colab meminta restart runtime setelah instal, restart lalu lanjut dari sel berikutnya.
!pip -q install open_clip_torch torchattacks datasets transformers sentencepiece sacremoses gradio scikit-learn pyyaml pandas matplotlib requests

In [ ]:
# ============================ KONFIGURASI TUNGGAL (R-01) ============================
# Semua parameter ada di sini. Disimpan ke config.yaml bersama hasil.
import os, sys, json, time, random, hashlib, platform, datetime, glob, shutil, re, copy, math, zipfile
import numpy as np, pandas as pd

CFG = dict(
    seeds=[0, 1, 2],                 # R-02: ditetapkan sebelum eksperimen, minimal 3
    data_seed=2024,                  # seed split validasi dan subset (tidak ikut berubah antar model)
    # --- data
    data_source='osf',               # 'osf' (resmi) atau 'hf' (cadangan, NFR-08)
    val_frac=0.10,                   # fraksi CITRA dari latih untuk validasi (split resmi tidak punya validasi)
    # --- model
    hf_id='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224',
    ctx_len=256, img_size=224,
    m1_epochs=40, m1_lr=1e-3, m1_bs=64, m1_hidden=512, m1_dropout=0.3, m1_wd=1e-4,
    # --- serangan citra (k/255)
    eps_list=[1, 2, 4, 8],
    eps_extra=[0.5],                 # dipakai otomatis bila RA(1/255) <= collapse_thr (FR-08)
    collapse_thr=0.25,
    attack_bs=32,
    pgd_steps=10, pgd_alpha_ratio=2.5,           # alpha = ratio * eps / steps
    safemed=dict(eps=0.01, alpha=0.002, steps=5),  # FR-09, = 2.55/255
    aa_version='standard',           # APGD-CE, APGD-T, FAB-T, Square
    aa_n_samples=None,               # None = semua; isi angka untuk subset bila terlalu lambat (NFR-02)
    square_queries=1000,
    # --- pemeriksaan gradient masking (FR-12) dan sanity (R-08)
    gm_eps_large=64, gm_x2_eps=[4, 8], gm_tol=0.02, gm_zero_thr=0.05,
    auto_quarantine=True,            # R-09: hasil gagal sanity dipindah ke results_rejected/
    # --- teks
    text_kinds=['typo', 'paraphrase', 'id'],
    typo_rate=0.15, text_seed=1234, mt_model='Helsinki-NLP/opus-mt-en-id',
    combo_attacks=['PGD'], combo_eps=[2, 4, 8],
    # --- Model 2 (TRADES)
    beta_grid=[1, 3, 6], beta_sel_epochs=2, beta=None,   # beta=None -> dipilih di validasi (R-06)
    m2_epochs=5, m2_bs=16, m2_lr_enc=1e-5, m2_lr_head=1e-4, m2_wd=1e-4, m2_amp=True,
    trades_eps=4, trades_steps=3, trades_step_ratio=2.5, trades_noise=1e-3,   # eps dalam k/255
    val_pgd_eps=4, val_pgd_steps=10,
    m2_save_all_epochs=True,         # satu berkas per epoch (PRD 9.2); matikan bila Drive penuh
    run_control=False,               # model kontrol: encoder dibuka tanpa TRADES (beta=0)
    max_clean_drop=0.05,             # K2 (usulan)
    # --- bootstrap
    boot_B=1000,
    launch_demo=True,
)

ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/vqa_robust'
elif ON_KAGGLE:
    ROOT = '/kaggle/working/vqa_robust'
else:
    ROOT = os.path.abspath('./vqa_robust')
DATA, CK, RES, REJ, FIG = [f'{ROOT}/{d}' for d in ['data', 'ckpt', 'results', 'results_rejected', 'figs']]
for d in [DATA, CK, RES, REJ, FIG]:
    os.makedirs(d, exist_ok=True)

import yaml
with open(f'{ROOT}/config.yaml', 'w') as f:
    yaml.safe_dump(CFG, f, sort_keys=True)
CFG_HASH = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]
print('ROOT =', ROOT, '| config hash =', CFG_HASH)

In [ ]:
# ============================ LINGKUNGAN, SEED, LOG (R-02, R-12, NFR-03) ============================
import torch, torch.nn as nn, torch.nn.functional as F
import importlib.metadata as md_
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ver(p):
    try: return md_.version(p)
    except Exception: return 'n/a'

ENV = dict(date=str(datetime.datetime.now()), python=platform.python_version(), torch=torch.__version__,
           cuda=torch.version.cuda, gpu=(torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU'),
           **{p: ver(p) for p in ['open_clip_torch', 'torchattacks', 'transformers', 'datasets', 'gradio', 'scikit-learn']},
           config_hash=CFG_HASH)
try:
    nb_path = [p for p in glob.glob('*.ipynb')]
    ENV['notebook_hash'] = hashlib.sha256(open(nb_path[0], 'rb').read()).hexdigest()[:12] if nb_path else 'n/a'
except Exception:
    ENV['notebook_hash'] = 'n/a'
with open(f'{ROOT}/env_{datetime.date.today()}.json', 'w') as f:
    json.dump(ENV, f, indent=1)
print(json.dumps(ENV, indent=1))
if DEV == 'cpu':
    print('PERINGATAN: GPU tidak terdeteksi. Aktifkan GPU T4 (NFR-01).')
set_seed(CFG['seeds'][0])

## 1. Data VQA-RAD (FR-01 sampai FR-04)

Sumber utama OSF (`osf.io/89kps`). Mirror Hugging Face hanya cadangan dan dicatat di `data_source.json` (NFR-08).
Split resmi dibaca dari kolom `phrase_type` (awalan `test` = uji). Citra diubah ke 224x224 dan disimpan di rentang [0,1]. **Normalisasi dilakukan di dalam model** (FR-04, R-04).

In [ ]:
import requests
from PIL import Image
import torchvision.transforms.functional as TF

def norm_ans(a): return str(a).strip().lower()

def load_osf():
    zpath = f'{DATA}/osf_vqa_rad.zip'
    if not os.path.exists(zpath):
        url = 'https://files.osf.io/v1/resources/89kps/providers/osfstorage/?zip='
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(zpath, 'wb') as f:
                for ch in r.iter_content(1 << 20): f.write(ch)
    ex = f'{DATA}/osf'
    if not os.path.isdir(ex):
        zipfile.ZipFile(zpath).extractall(ex)
        # beberapa rilis menaruh folder citra di dalam zip lagi
        for z in glob.glob(f'{ex}/**/*.zip', recursive=True):
            zipfile.ZipFile(z).extractall(os.path.splitext(z)[0])
    js = [p for p in glob.glob(f'{ex}/**/*.json', recursive=True)]
    js = sorted(js, key=os.path.getsize)[-1]
    items = json.load(open(js, encoding='utf-8'))
    paths = {}
    for p in glob.glob(f'{ex}/**/*', recursive=True):
        if p.lower().endswith(('.jpg', '.jpeg', '.png')): paths[os.path.basename(p)] = p
    rows, imgs = [], {}
    for it in items:
        name = it['image_name']
        if name not in paths: continue
        imgs.setdefault(name, Image.open(paths[name]).convert('RGB'))
        a = norm_ans(it['answer'])
        rows.append(dict(img_id=name, question=str(it['question']).strip(), answer=a,
                         atype=str(it['answer_type']).lower(),
                         split='test' if str(it['phrase_type']).lower().startswith('test') else 'train'))
    return pd.DataFrame(rows), imgs, dict(source='OSF osf.io/89kps', file=js)

def load_hf():
    from datasets import load_dataset
    ds = load_dataset('flaviagiammarino/vqa-rad')
    rows, imgs = [], {}
    for split in ['train', 'test']:
        for r in ds[split]:
            im = r['image'].convert('RGB')
            iid = hashlib.md5(im.tobytes()).hexdigest()[:12]
            imgs.setdefault(iid, im)
            a = norm_ans(r['answer'])
            rows.append(dict(img_id=iid, question=str(r['question']).strip(), answer=a,
                             atype='closed' if a in ('yes', 'no') else 'open', split=split))
    return pd.DataFrame(rows), imgs, dict(source='HF mirror flaviagiammarino/vqa-rad (CADANGAN)')

try:
    if CFG['data_source'] == 'osf': df, PIL_IMGS, SRC = load_osf()
    else: raise RuntimeError('hf dipilih')
except Exception as e:
    print('OSF gagal/dilewati ->', repr(e)[:200], '\nMemakai mirror HF (cadangan).')
    df, PIL_IMGS, SRC = load_hf()
SRC['date'] = str(datetime.date.today())
json.dump(SRC, open(f'{ROOT}/data_source.json', 'w'), indent=1)
print(SRC)

# FR-01: hitung citra dan pasangan
df = df.drop_duplicates(subset=['img_id', 'question', 'answer', 'split']).reset_index(drop=True)
df['qid'] = np.arange(len(df))
print('Citra:', df.img_id.nunique(), '| Pasangan tanya-jawab:', len(df))
print(df.groupby(['split', 'atype']).size().unstack(fill_value=0))

In [ ]:
# FR-02/FR-03: pertanyaan tertutup ya/tidak, validasi per CITRA, distribusi kelas, ubah ukuran citra
df['yn'] = (df.atype == 'closed') & df.answer.isin(['yes', 'no'])
rng = np.random.default_rng(CFG['data_seed'])
tr_imgs = np.array(sorted(df[df.split == 'train'].img_id.unique()))
val_imgs = set(rng.choice(tr_imgs, size=int(round(CFG['val_frac'] * len(tr_imgs))), replace=False))
df['part'] = np.where(df.split == 'test', 'test', np.where(df.img_id.isin(val_imgs), 'val', 'train'))

img_ids = sorted(df.img_id.unique()); IMG_INDEX = {k: i for i, k in enumerate(img_ids)}
df['img_idx'] = df.img_id.map(IMG_INDEX)
S = CFG['img_size']
IMGS = torch.stack([TF.to_tensor(PIL_IMGS[k].resize((S, S), Image.BICUBIC)) for k in img_ids]).to(DEV)  # [0,1]
print('IMGS', tuple(IMGS.shape), float(IMGS.min()), float(IMGS.max()))

print('\nJumlah per bagian dan tipe:')
print(df.groupby(['part', 'yn']).size().unstack(fill_value=0).rename(columns={True: 'closed yes/no', False: 'lainnya'}))
print('\nDistribusi kelas ya/tidak (FR-03):')
dist = df[df.yn].groupby(['part', 'answer']).size().unstack(fill_value=0)
dist['majority_acc'] = dist.max(axis=1) / dist[['no', 'yes']].sum(axis=1)
print(dist)

# kosakata jawaban tetap dari bagian latih
vc = df[df.part == 'train'].answer.value_counts()
VOCAB = list(vc.index); A2I = {a: i for i, a in enumerate(VOCAB)}
NO, YES = A2I['no'], A2I['yes']
df['y'] = df.answer.map(lambda a: A2I.get(a, -1))              # -1 = di luar kosakata
df['y2'] = np.where(df.yn, (df.answer == 'yes').astype(int), -1)  # label 2 arah
print('Kosakata:', len(VOCAB), 'jawaban | majority baseline uji (yes/no):', round(dist.loc['test', 'majority_acc'], 4))

def T(a, dtype=torch.long): return torch.tensor(np.asarray(a), dtype=dtype, device=DEV)
Y, Y2, IMGIDX = T(df.y.values), T(df.y2.values), T(df.img_idx.values)
def idx_of(mask): return T(np.where(mask)[0])
idx_train, idx_val, idx_test = [idx_of(df.part == p) for p in ['train', 'val', 'test']]
idx_val_cl, idx_test_cl = idx_of((df.part == 'val') & df.yn), idx_of((df.part == 'test') & df.yn)
IMG_OF = df.img_id.values  # untuk bootstrap tingkat citra (FR-15)
print('uji total:', len(idx_test), '| uji ya/tidak:', len(idx_test_cl), '| val ya/tidak:', len(idx_val_cl))

In [ ]:
# ============================ CHECKPOINT CP0 ============================
def checkpoint(name, conds, hard=True):
    ok = all(c for _, c in conds)
    for msg, c in conds: print(('  [LULUS] ' if c else '  [GAGAL] ') + msg)
    print(f'=== {name}: ' + ('LULUS' if ok else 'GAGAL') + ' ===')
    if hard and not ok: raise AssertionError(f'{name} gagal. Perbaiki dulu sebelum lanjut (PRD 9.1).')

checkpoint('CP0 (data)', [
    ('jumlah citra dan pasangan terhitung', df.img_id.nunique() > 0 and len(df) > 0),
    (f'uji resmi = 451 pasangan (ditemukan {len(idx_test)}); bila beda, cek berkas OSF dan perbarui teks proposal (pertanyaan terbuka no. 3)', len(idx_test) == 451),
    ('distribusi kelas ya/tidak dihitung', dist.shape[0] >= 2),
    ('tidak ada citra yang sama di latih/val dan uji', len(set(df[df.part == 'test'].img_id) & set(df[df.part != 'test'].img_id)) == 0),
], hard=False)   # hard=False: selisih 451 tidak menghentikan notebook, tapi WAJIB dicatat

## 2. Model 1: BiomedCLIP beku + kepala fusi (FR-05)

Encoder citra dan teks BiomedCLIP beku. Embedding citra (512) dan teks (512) digabung (konkatenasi) lalu masuk kepala MLP yang menghasilkan kelas jawaban dari kosakata tetap, loss cross-entropy. Hanya kepala yang dilatih, 3 seed.
Embedding teks pertanyaan dihitung sekali dan disimpan (encoder teks beku di semua model).

In [ ]:
import open_clip, torchattacks as ta
clip_model, _, _ = open_clip.create_model_and_transforms(CFG['hf_id'])
tokenizer = open_clip.get_tokenizer(CFG['hf_id'])
clip_model = clip_model.to(DEV).eval()
for p in clip_model.parameters(): p.requires_grad_(False)
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)

@torch.no_grad()
def embed_text(questions, bs=128):
    out = []
    for i in range(0, len(questions), bs):
        tok = tokenizer(list(questions[i:i + bs]), context_length=CFG['ctx_len']).to(DEV)
        out.append(clip_model.encode_text(tok).float())
    return torch.cat(out)

@torch.no_grad()
def embed_images(x, bs=64):
    m, s = MEAN.to(DEV), STD.to(DEV)
    return torch.cat([clip_model.visual((x[i:i + bs] - m) / s).float() for i in range(0, len(x), bs)])

Q0 = embed_text(df.question.tolist())          # embedding pertanyaan asli
FEATS = embed_images(IMGS)                     # fitur citra beku (hanya untuk melatih Model 1 lebih cepat)
EMB = FEATS.shape[1]
print('Q0', tuple(Q0.shape), 'FEATS', tuple(FEATS.shape))

def make_head(n_cls, emb=None):
    emb = emb or EMB
    return nn.Sequential(nn.Linear(2 * emb, CFG['m1_hidden']), nn.ReLU(), nn.Dropout(CFG['m1_dropout']),
                         nn.Linear(CFG['m1_hidden'], n_cls))

class VQANet(nn.Module):
    # citra [0,1] + embedding pertanyaan -> logit kosakata. Normalisasi ada DI DALAM model (R-04).
    def __init__(self, visual, head, freeze_visual):
        super().__init__()
        self.visual, self.head, self.freeze_visual = visual, head, freeze_visual
        self.register_buffer('mean', MEAN.clone()); self.register_buffer('std', STD.clone())
        for p in self.visual.parameters(): p.requires_grad_(not freeze_visual)
    def train(self, mode=True):
        super().train(mode)
        if self.freeze_visual: self.visual.eval()
        return self
    def img_feat(self, x): return self.visual((x - self.mean) / self.std).float()
    def forward(self, x, q): return self.head(torch.cat([self.img_feat(x), q], 1))

class Wrap2(nn.Module):
    # pembungkus untuk pustaka serangan: citra -> 2 logit [no, yes], embedding pertanyaan ditahan tetap
    def __init__(self, net): super().__init__(); self.net = net; self.q = None
    def forward(self, x): return self.net(x, self.q)[:, [NO, YES]]

CKPT_NAME = {'M1': 'model1', 'M2': 'model2', 'M2ctl': 'model2ctl'}
def get_net(tag, seed):
    path = f'{CK}/{CKPT_NAME[tag]}_seed{seed}_best.pt'
    st = torch.load(path, map_location=DEV)
    if tag == 'M1':
        net = VQANet(clip_model.visual, make_head(len(VOCAB)), True)
    else:
        net = VQANet(copy.deepcopy(clip_model.visual), make_head(len(VOCAB)), False)
        net.visual.load_state_dict(st['visual'])
    net.head.load_state_dict(st['head'])
    return net.to(DEV).eval()

In [ ]:
# ============================ PELATIHAN MODEL 1 (hanya kepala fusi), dengan resume ============================
def head_logits(head, idx, bs=512):
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]
            out.append(head(torch.cat([FEATS[IMGIDX[b]], Q0[b]], 1)))
    return torch.cat(out)

def val_em(head, idx):
    head.eval(); lg = head_logits(head, idx)
    return float((lg.argmax(1) == Y[idx]).float().mean())

def train_model1(seed):
    best_p = f'{CK}/model1_seed{seed}_best.pt'
    if os.path.exists(best_p):
        print(f'[M1 seed {seed}] sudah ada, dilewati'); return
    set_seed(seed)
    head = make_head(len(VOCAB)).to(DEV)
    opt = torch.optim.AdamW(head.parameters(), lr=CFG['m1_lr'], weight_decay=CFG['m1_wd'])
    start, best = 0, -1.0
    rp = f'{CK}/model1_seed{seed}_resume.pt'
    if os.path.exists(rp):
        st = torch.load(rp, map_location=DEV)
        head.load_state_dict(st['head']); opt.load_state_dict(st['opt']); start, best = st['epoch'] + 1, st['best']
        print(f'[M1 seed {seed}] lanjut dari epoch {start}')
    for ep in range(start, CFG['m1_epochs']):
        head.train()
        perm = idx_train[torch.randperm(len(idx_train), device=DEV)]
        for i in range(0, len(perm), CFG['m1_bs']):
            b = perm[i:i + CFG['m1_bs']]
            loss = F.cross_entropy(head(torch.cat([FEATS[IMGIDX[b]], Q0[b]], 1)), Y[b])
            opt.zero_grad(); loss.backward(); opt.step()
        v = val_em(head, idx_val)
        ck = dict(head=head.state_dict(), epoch=ep, seed=seed, val_em=v, hf_id=CFG['hf_id'],
                  open_clip=ver('open_clip_torch'), cfg_hash=CFG_HASH)
        torch.save(ck, f'{CK}/model1_seed{seed}_ep{ep}.pt')
        if v > best:
            best = v; torch.save(ck, f'{CK}/model1_seed{seed}_bestsofar.pt')
        torch.save(dict(head=head.state_dict(), opt=opt.state_dict(), epoch=ep, best=best), rp)
        if ep % 5 == 0 or ep == CFG['m1_epochs'] - 1: print(f'[M1 seed {seed}] ep {ep} loss {loss.item():.3f} val_em {v:.3f} best {best:.3f}')
    shutil.copy(f'{CK}/model1_seed{seed}_bestsofar.pt', best_p)   # terbaik di VALIDASI, bukan uji (R-06)

for s in CFG['seeds']:
    train_model1(s)

In [ ]:
# ============================ CP1: EVALUASI BERSIH TIDAK BOLEH LEBIH BURUK DARI MENEBAK MAYORITAS ============================
maj = float(dist.loc['test', 'majority_acc'])
accs = {}
for s in CFG['seeds']:
    net = get_net('M1', s)
    with torch.no_grad():
        wrap = Wrap2(net); idx = idx_test_cl
        wrap.q = Q0[idx]; pr = wrap(IMGS[IMGIDX[idx]]).argmax(1)
    accs[s] = float((pr == Y2[idx]).float().mean())
print('Akurasi bersih Model 1 (uji ya/tidak) per seed:', accs, '| majority:', round(maj, 4))
checkpoint('CP1 (Model 1 sudah belajar)', [
    ('akurasi bersih rata-rata lebih baik dari menebak kelas mayoritas', np.mean(list(accs.values())) > maj),
    ('semua seed punya checkpoint terbaik', all(os.path.exists(f'{CK}/model1_seed{s}_best.pt') for s in CFG['seeds'])),
], hard=False)

## 3. Serangan citra: evaluator, penamaan hasil, smoke test (FR-08, FR-09, NFR-02)

Setiap kombinasi *(model, serangan, ε, seed, jenis teks)* disimpan ke `results/*.json` termasuk prediksi per sampel dan confidence (PRD 9.2). Bila berkas sudah ada, kombinasi dilewati sehingga run terputus bisa dilanjutkan.
Citra ada di [0,1], hasil perturbasi dipotong ke [0,1], model dalam mode `eval` saat menyerang (R-03, R-04). ε ditulis k/255 (R-05).

In [ ]:
def make_attack(wrap, name, eps255, seed):
    e = eps255 / 255.0
    if name == 'FGSM': return ta.FGSM(wrap, eps=e)
    if name in ('PGD', 'PGDx2'):
        st = CFG['pgd_steps'] * (2 if name == 'PGDx2' else 1)
        return ta.PGD(wrap, eps=e, alpha=CFG['pgd_alpha_ratio'] * e / st, steps=st, random_start=True)
    if name == 'PGDsafemed':
        c = CFG['safemed']; return ta.PGD(wrap, eps=c['eps'], alpha=c['alpha'], steps=c['steps'], random_start=True)
    if name == 'AutoAttack':
        return ta.AutoAttack(wrap, norm='Linf', eps=e, version=CFG['aa_version'], n_classes=2, seed=seed)
    if name == 'Square':
        return ta.Square(wrap, norm='Linf', eps=e, n_queries=CFG['square_queries'], n_restarts=1, seed=seed)
    raise ValueError(name)

def run_closed(net, idx, attack, eps255, q_all, seed, q_orig=None):
    # Evaluasi pertanyaan ya/tidak. q_all = embedding pertanyaan yang dipakai (asli atau diubah).
    # Mengembalikan prediksi pada: (orig) citra+teks asli, (clean) citra asli+teks yang dipakai, (adv) citra diserang+teks yang dipakai.
    q_orig = Q0 if q_orig is None else q_orig
    net.eval(); wrap = Wrap2(net)
    atk = None if (attack == 'Clean' or eps255 == 0) else make_attack(wrap, attack, eps255, seed)
    o = {k: [] for k in ['pred_orig', 'pred_clean', 'pred_adv', 'p_clean', 'p_adv']}; linf = 0.0
    for i in range(0, len(idx), CFG['attack_bs']):
        b = idx[i:i + CFG['attack_bs']]
        x, y = IMGS[IMGIDX[b]], Y2[b]
        with torch.no_grad():
            wrap.q = q_orig[b]; lo = wrap(x)
            wrap.q = q_all[b]; lc = wrap(x)
        wrap.q = q_all[b]
        if atk is None: xa = x
        else:
            xa = atk(x, y); linf = max(linf, float((xa - x).abs().max()))
            assert xa.min() >= 0 and xa.max() <= 1, 'perturbasi keluar dari [0,1]'
        with torch.no_grad(): la = wrap(xa)
        o['pred_orig'].append(lo.argmax(1)); o['pred_clean'].append(lc.argmax(1)); o['pred_adv'].append(la.argmax(1) if atk is not None else lc.argmax(1))
        o['p_clean'].append(F.softmax(lc, 1)[:, 1]); o['p_adv'].append(F.softmax(la, 1)[:, 1] if atk is not None else F.softmax(lc, 1)[:, 1])
    out = {k: torch.cat(v).cpu().numpy() for k, v in o.items()}
    out['y'] = Y2[idx].cpu().numpy(); out['linf'] = linf
    return out

def rkey(tag, attack, eps, seed, text='none'):
    return f'{tag}_{attack}_eps{eps:g}_seed{seed}' + ('' if text == 'none' else f'_txt-{text}')

def aa_subset(idx):
    n = CFG['aa_n_samples']
    if not n or n >= len(idx): return idx
    r = np.random.default_rng(CFG['data_seed']); sel = np.sort(r.choice(len(idx), n, replace=False))
    return idx[torch.as_tensor(sel, device=idx.device)]

QT = {}   # embedding pertanyaan yang diubah, diisi di bagian 4

def do_run(tag, net, attack, eps, seed, text='none'):
    k = rkey(tag, attack, eps, seed, text)
    if os.path.exists(f'{RES}/{k}.json') or os.path.exists(f'{REJ}/{k}.json'): return
    set_seed(seed)
    idx = aa_subset(idx_test_cl) if attack == 'AutoAttack' else idx_test_cl
    q_all = Q0 if text == 'none' else QT[text]
    t0 = time.time()
    o = run_closed(net, idx, attack, eps, q_all, seed)
    dt = time.time() - t0
    ii = idx.cpu().numpy()
    data = {k_: v.tolist() for k_, v in o.items() if k_ != 'linf'}
    data['img'] = IMG_OF[ii].tolist(); data['qid'] = df.qid.values[ii].tolist()
    meta = dict(tag=tag, attack=attack, eps=eps, seed=seed, text=text, n=len(ii), gpu_seconds=round(dt, 1),
                linf=o['linf'] * 255, label='uji robustness' if text != 'none' else 'adversarial', cfg_hash=CFG_HASH,
                torch=torch.__version__, torchattacks=ver('torchattacks'), open_clip=ver('open_clip_torch'))
    json.dump(dict(meta=meta, data=data), open(f'{RES}/{k}.json', 'w'))
    acc = np.mean(np.array(o['pred_adv']) == o['y'])
    print(f'{k:60s} acc={acc:.3f} n={len(ii)} {dt:5.0f}s Linf={o["linf"]*255:.2f}/255')

In [ ]:
# ============================ SMOKE TEST 20 SAMPEL (R-07, NFR-02) ============================
# Menaksir waktu GPU sebelum run penuh. Ulangi setiap kali kode serangan atau model berubah.
def smoke_test(tag='M1', seed=None, n=20, eps=4):
    seed = CFG['seeds'][0] if seed is None else seed
    try: net = get_net(tag, seed); note = ''
    except Exception:
        net = VQANet(clip_model.visual, make_head(len(VOCAB)), True).to(DEV).eval(); note = ' (kepala acak: waktu AutoAttack bisa meleset, ulangi setelah Model 1 terlatih)'
    idx = idx_test_cl[:n]; rows = []
    for atk in ['FGSM', 'PGD', 'AutoAttack', 'Square']:
        if DEV == 'cuda': torch.cuda.synchronize()
        t = time.time(); run_closed(net, idx, atk, eps, Q0, seed)
        if DEV == 'cuda': torch.cuda.synchronize()
        per = (time.time() - t) / n
        ns = len(aa_subset(idx_test_cl)) if atk == 'AutoAttack' else len(idx_test_cl)
        n_eps = len(CFG['eps_list']) + (len(CFG['gm_x2_eps']) if atk == 'PGD' else 0)
        rows.append(dict(serangan=atk, detik_per_sampel=round(per, 3), jam_total_per_model=round(per * ns * n_eps * len(CFG['seeds']) / 3600, 2)))
    print('Estimasi waktu GPU' + note); print(pd.DataFrame(rows).to_string(index=False))
    print('Bila total terlalu besar: isi CFG["aa_n_samples"] (subset) atau persempit eps_list (NFR-02).')
smoke_test()

## 4. Perturbasi teks (FR-10, FR-11)

Tiga jenis: **typo**, **parafrase** (aturan templat, tidak dicari secara adversarial), dan **terjemahan ke bahasa Indonesia** (MarianMT). Semuanya berlabel **"uji robustness"**, bukan "adversarial" (R-11).
Hanya pertanyaan uji ya/tidak yang diubah. Untuk serangan gabungan, embedding pertanyaan dihitung ulang dari teks yang diubah sebelum serangan citra dijalankan.
Catatan: encoder teks BiomedCLIP berbahasa Inggris, jadi penurunan pada teks Indonesia diperkirakan besar.

In [ ]:
def typo(text, rate, rnd):
    ws = text.split(' '); changed = False
    for i, w in enumerate(ws):
        core = re.sub(r'\W', '', w)
        if len(core) > 3 and rnd.random() < rate:
            j = rnd.randrange(1, len(w) - 1) if len(w) > 2 else 0
            op = rnd.choice(['swap', 'drop', 'dup'])
            if op == 'swap' and j + 1 < len(w): w = w[:j] + w[j + 1] + w[j] + w[j + 2:]
            elif op == 'drop': w = w[:j] + w[j + 1:]
            else: w = w[:j] + w[j] + w[j:]
            ws[i] = w; changed = True
    if not changed:   # pastikan minimal satu perubahan
        k = max(range(len(ws)), key=lambda i: len(ws[i])); w = ws[k]
        if len(w) > 3: ws[k] = w[:1] + w[2] + w[1] + w[3:]
    return ' '.join(ws)

PARA_RULES = [
    (r'^is there (.+)\?$', r'Can you see \1 in this image?'),
    (r'^are there (.+)\?$', r'Can you see \1 in this image?'),
    (r'^does this image show (.+)\?$', r'Is \1 visible in this image?'),
    (r'^does the image show (.+)\?$', r'Is \1 visible in the image?'),
    (r'^is the (.+) normal\?$', r'Does the \1 look normal?'),
    (r'^is this (.+)\?$', r'Would you say this is \1?'),
    (r'^is (.+) present\?$', r'Can you see \1 in this image?'),
    (r'^are (.+) present\?$', r'Can you see \1 in this image?'),
    (r'^is (.+) seen\?$', r'Can \1 be seen?'),
    (r'^is the (.+) (enlarged|abnormal|dilated|displaced)\?$', r'Does the \1 appear \2?'),
]
SYN = [(r'\bimage\b', 'scan'), (r'\bshow\b', 'display'), (r'\bvisible\b', 'apparent'), (r'\babnormal\b', 'not normal'),
       (r'\bpresent\b', 'evident'), (r'\bseen\b', 'visible')]
def paraphrase(q):
    s = q.strip(); low = s.lower()
    for pat, rep in PARA_RULES:
        if re.match(pat, low):
            return re.sub(pat, rep, low).capitalize()
    for pat, rep in SYN:
        new = re.sub(pat, rep, low)
        if new != low: return new.capitalize()
    return 'Looking at this scan, ' + low[:1].lower() + low[1:]

def translate_id(texts):
    from transformers import MarianMTModel, MarianTokenizer
    tok = MarianTokenizer.from_pretrained(CFG['mt_model']); mt = MarianMTModel.from_pretrained(CFG['mt_model']).to(DEV).eval()
    out = []
    for i in range(0, len(texts), 32):
        b = tok(texts[i:i + 32], return_tensors='pt', padding=True, truncation=True).to(DEV)
        with torch.no_grad(): g = mt.generate(**b, num_beams=4, max_length=128)
        out += tok.batch_decode(g, skip_special_tokens=True)
    del mt; torch.cuda.empty_cache(); return out

import random as _r
tpath = f'{DATA}/text_variants.json'
ii = idx_test_cl.cpu().numpy(); orig_q = df.question.values[ii].tolist()
if os.path.exists(tpath) and json.load(open(tpath)).get('qids') == df.qid.values[ii].tolist():
    VARS = json.load(open(tpath))['v']
else:
    VARS = dict(typo=[typo(q, CFG['typo_rate'], _r.Random(CFG['text_seed'] + int(i))) for q, i in zip(orig_q, df.qid.values[ii])],
                paraphrase=[paraphrase(q) for q in orig_q], id=translate_id(orig_q))
    json.dump(dict(qids=df.qid.values[ii].tolist(), v=VARS), open(tpath, 'w'), ensure_ascii=False, indent=0)
for k in CFG['text_kinds']:
    QT[k] = Q0.clone(); QT[k][idx_test_cl] = embed_text(VARS[k])
print('Contoh perturbasi teks (UJI ROBUSTNESS, bukan adversarial):')
for j in range(4): print(' asli      :', orig_q[j]); [print(f' {k:10s}:', VARS[k][j]) for k in CFG['text_kinds']]; print()

## 5. Skenario 1 sampai 4 pada Model 1

Satu fungsi `run_scenarios(tag)` menjalankan semuanya untuk model dan seed tertentu, dipakai ulang untuk Model 2 di bagian 6.
- Skenario 1: data bersih (`Clean`).
- Skenario 2: FGSM, PGD, AutoAttack pada ε = 1, 2, 4, 8 per 255, plus PGD ala SafeMed-R1 (ε=0,01 ≈ 2,55/255, α=0,002, 5 langkah). Titik ε=0,5/255 ditambah otomatis bila Model 1 runtuh pada 1/255.
- Skenario 3: typo, parafrase, terjemahan Indonesia.
- Skenario 4: serangan citra gabungan dengan perubahan teks.
- Tambahan untuk pemeriksaan gradient masking (FR-12): Square Attack, PGD dengan iterasi digandakan, PGD pada ε=64/255.

In [ ]:
def load_arr(path):
    d = json.load(open(path)); return d['meta'], {k: np.array(v) for k, v in d['data'].items()}

def run_scenarios(tag, seeds=None):
    for s in (seeds or CFG['seeds']):
        net = get_net(tag, s)
        do_run(tag, net, 'Clean', 0, s)                                              # skenario 1
        for atk in ['FGSM', 'PGD', 'AutoAttack']:                                    # skenario 2
            for e in CFG['eps_list']: do_run(tag, net, atk, e, s)
        do_run(tag, net, 'PGDsafemed', CFG['safemed']['eps'] * 255, s)               # FR-09
        p1 = f'{RES}/{rkey(tag, "PGD", 1, s)}.json'
        if os.path.exists(p1):                                                       # FR-08: titik ε kecil bila runtuh
            _, a = load_arr(p1)
            if np.mean(a['pred_adv'] == a['y']) <= CFG['collapse_thr']:
                print(f'[{tag} seed {s}] runtuh pada 1/255 -> tambah ε', CFG['eps_extra'])
                for atk in ['FGSM', 'PGD', 'AutoAttack']:
                    for e in CFG['eps_extra']: do_run(tag, net, atk, e, s)
        for e in CFG['eps_list']: do_run(tag, net, 'Square', e, s)                   # FR-12 (a)
        for e in CFG['gm_x2_eps']: do_run(tag, net, 'PGDx2', e, s)                   # FR-12 (d)
        do_run(tag, net, 'PGD', CFG['gm_eps_large'], s)                              # FR-12 (c)
        for t in CFG['text_kinds']: do_run(tag, net, 'Clean', 0, s, text=t)          # skenario 3
        for t in CFG['text_kinds']:                                                  # skenario 4
            for atk in CFG['combo_attacks']:
                for e in CFG['combo_eps']: do_run(tag, net, atk, e, s, text=t)
        del net
        if DEV == 'cuda': torch.cuda.empty_cache()
    print(f'run_scenarios({tag}) selesai')

run_scenarios('M1')

In [ ]:
# Pertanyaan terbuka: exact match pada kosakata penuh, data bersih, dilaporkan terpisah (FR-16)
def eval_open(tag):
    for s in CFG['seeds']:
        p = f'{RES}/OPEN_{tag}_seed{s}.json'
        if os.path.exists(p): continue
        net = get_net(tag, s); idx = idx_of((df.part == 'test') & (df.atype == 'open'))
        with torch.no_grad(): lg = net(IMGS[IMGIDX[idx]], Q0[idx])
        c = (lg.argmax(1) == Y[idx]).cpu().numpy().astype(int)
        json.dump(dict(correct=c.tolist(), img=IMG_OF[idx.cpu().numpy()].tolist()), open(p, 'w'))
        del net
eval_open('M1')

In [ ]:
# ============================ SANITY OTOMATIS (R-08, R-09) DAN CP2/CP3 ============================
def load_all():
    recs = {}
    for p in glob.glob(f'{RES}/M*_seed*.json'):
        m, a = load_arr(p)
        recs.setdefault((m['tag'], m['attack'], float(m['eps']), m['text']), {})[m['seed']] = (m, a, p)
    return recs

def acc_of(a): return float(np.mean(a['pred_adv'] == a['y']))
def mean_acc(recs, tag, atk, eps, text='none'):
    r = recs.get((tag, atk, float(eps), text));  return None if not r else float(np.mean([acc_of(v[1]) for v in r.values()]))

def quarantine(files, reason):
    log = f'{REJ}/rejected_log.json'; L = json.load(open(log)) if os.path.exists(log) else []
    for p in files:
        if os.path.exists(p): shutil.move(p, f'{REJ}/{os.path.basename(p)}'); L.append(dict(file=os.path.basename(p), reason=reason))
    json.dump(L, open(log, 'w'), indent=1)

def sanity(tag, verbose=True):
    recs = load_all(); tol = CFG['gm_tol']; bad = []   # bad: (files, alasan)
    clean = mean_acc(recs, tag, 'Clean', 0)
    for atk in ['FGSM', 'PGD', 'AutoAttack', 'Square']:
        es = sorted(e for (t, a, e, tx) in recs if t == tag and a == atk and tx == 'none')
        prev = clean
        for e in es:
            ra = mean_acc(recs, tag, atk, e); files = [v[2] for v in recs[(tag, atk, e, 'none')].values()]
            # (a) konsistensi: akurasi asli di berkas serangan = akurasi bersih
            po = np.mean([np.mean(v[1]['pred_orig'] == v[1]['y']) for v in recs[(tag, atk, e, 'none')].values()])
            if clean is not None and abs(po - clean) > 1e-6 and atk != 'AutoAttack':
                bad.append((files, f'(a) akurasi ε=0 di berkas {atk} {e:g} tidak sama dengan akurasi bersih'))
            if clean is not None and ra > clean + tol: bad.append((files, f'(b) {atk} ε={e:g}: RA {ra:.3f} > bersih {clean:.3f}'))
            elif prev is not None and ra > prev + tol and atk != 'AutoAttack': bad.append((files, f'(c) {atk} ε={e:g}: RA naik saat ε naik ({prev:.3f} -> {ra:.3f})'))
            prev = ra
    if verbose:
        for f, r in bad: print('SANITY GAGAL:', r)
        if not bad: print(f'[{tag}] sanity (a)-(c) lulus')
    return bad

def apply_sanity(tag):
    bad = sanity(tag)
    if CFG['auto_quarantine']:
        for files, reason in bad: quarantine(files, reason)
        if bad: print('Berkas gagal dipindah ke', REJ, '(tidak dihapus). Jalankan ulang run_scenarios bila mau memeriksa penyebabnya.')
    return bad
apply_sanity('M1')

recs = load_all()
checkpoint('CP2 (serangan citra Model 1)', [
    ('skenario 1 ada untuk semua seed', all((('M1', 'Clean', 0.0, 'none') in recs) and s in recs[('M1', 'Clean', 0.0, 'none')] for s in CFG['seeds'])),
    ('FGSM/PGD/AutoAttack ada untuk semua ε dan seed', all(len(recs.get(('M1', a, float(e), 'none'), {})) == len(CFG['seeds']) for a in ['FGSM', 'PGD', 'AutoAttack'] for e in CFG['eps_list'])),
    ('konfigurasi PGD ala SafeMed-R1 ada', ('M1', 'PGDsafemed', round(CFG['safemed']['eps'] * 255, 2), 'none') in recs or any(k[1] == 'PGDsafemed' for k in recs)),
    ('sanity (a)-(c) lulus', len(sanity('M1', verbose=False)) == 0),
], hard=False)
checkpoint('CP3 (teks dan gabungan Model 1)', [
    ('skenario 3 ada untuk semua jenis teks', all(('M1', 'Clean', 0.0, t) in recs for t in CFG['text_kinds'])),
    ('skenario 4 ada', all(('M1', a, float(e), t) in recs for a in CFG['combo_attacks'] for e in CFG['combo_eps'] for t in CFG['text_kinds'])),
], hard=False)

## 6. Model 2: adversarial fine-tuning TRADES (FR-06) dan pemeriksaan gradient masking (FR-12)

- Encoder citra dibuka, encoder teks beku (embedding pertanyaan tetap), batch kecil, mixed precision (FR-06).
- Inisialisasi kepala dari Model 1 dengan seed yang sama. Perturbasi **hanya pada citra**.
- Loss TRADES = CE(natural) + β · KL(p(x) || p(x_adv)), mengikuti `trades_loss()` resmi (MIT).
- β dipilih pada **data validasi** (seed pertama), bukan set uji (NFR-05, R-06). Checkpoint terbaik dipilih dari skor validasi = rata-rata akurasi bersih dan robust (PGD).
- Setiap epoch disimpan, dan pelatihan bisa dilanjutkan (resume).

In [ ]:
def trades_loss(net, x, q, y, beta, eps, step, steps, noise):
    # Mengikuti yaodongyu/TRADES trades.py (MIT), diadaptasi untuk masukan (citra, embedding pertanyaan).
    if beta == 0:                       # model kontrol: fine-tune encoder tanpa TRADES
        net.train(); return F.cross_entropy(net(x, q).float(), y)
    net.eval()
    with torch.no_grad(): p_nat = F.softmax(net(x, q).float(), dim=1)
    x_adv = (x + noise * torch.randn_like(x)).clamp(0, 1)
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        with torch.enable_grad():
            kl = F.kl_div(F.log_softmax(net(x_adv, q).float(), dim=1), p_nat, reduction='sum') * 128.0   # skala: cegah underflow fp16
            g, = torch.autograd.grad(kl, x_adv)
        x_adv = x_adv.detach() + step * torch.nan_to_num(g).sign()
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0, 1)
    net.train()
    ln = net(x, q).float(); la = net(x_adv.detach(), q).float()
    return F.cross_entropy(ln, y) + beta * F.kl_div(F.log_softmax(la, 1), F.softmax(ln, 1), reduction='batchmean')

def val_metrics(net):
    # akurasi bersih dan robust (PGD) pada validasi ya/tidak; hanya untuk memilih beta dan checkpoint
    set_seed(CFG['data_seed'])
    wrap = Wrap2(net.eval()); atk = ta.PGD(wrap, eps=CFG['val_pgd_eps'] / 255, alpha=CFG['pgd_alpha_ratio'] * CFG['val_pgd_eps'] / 255 / CFG['val_pgd_steps'],
                                          steps=CFG['val_pgd_steps'], random_start=True)
    c = r = 0
    for i in range(0, len(idx_val_cl), 32):
        b = idx_val_cl[i:i + 32]; x, y = IMGS[IMGIDX[b]], Y2[b]; wrap.q = Q0[b]
        with torch.no_grad(): c += int((wrap(x).argmax(1) == y).sum())
        xa = atk(x, y)
        with torch.no_grad(): r += int((wrap(xa).argmax(1) == y).sum())
    return c / len(idx_val_cl), r / len(idx_val_cl)

def train_model2(seed, beta, epochs, name, base_seed=None):
    best_p = f'{CK}/{name}_best.pt'
    if os.path.exists(best_p): print(f'[{name}] sudah ada, dilewati'); return best_p
    set_seed(seed)
    base = torch.load(f'{CK}/model1_seed{seed if base_seed is None else base_seed}_best.pt', map_location=DEV)
    net = VQANet(copy.deepcopy(clip_model.visual), make_head(len(VOCAB)), False).to(DEV)
    net.head.load_state_dict(base['head'])
    opt = torch.optim.AdamW([dict(params=net.visual.parameters(), lr=CFG['m2_lr_enc']), dict(params=net.head.parameters(), lr=CFG['m2_lr_head'])], weight_decay=CFG['m2_wd'])
    amp = CFG['m2_amp'] and DEV == 'cuda'; scaler = torch.cuda.amp.GradScaler(enabled=amp)
    start, best = 0, -1.0; rp = f'{CK}/{name}_resume.pt'
    if os.path.exists(rp):
        st = torch.load(rp, map_location=DEV); net.visual.load_state_dict(st['visual']); net.head.load_state_dict(st['head'])
        opt.load_state_dict(st['opt']); scaler.load_state_dict(st['scaler']); start, best = st['epoch'] + 1, st['best']; print(f'[{name}] lanjut dari epoch {start}')
    eps = CFG['trades_eps'] / 255; steps = CFG['trades_steps']; stp = CFG['trades_step_ratio'] * eps / steps
    for ep in range(start, epochs):
        net.train(); t0 = time.time()
        perm = idx_train[torch.randperm(len(idx_train), device=DEV)]
        for i in range(0, len(perm), CFG['m2_bs']):
            b = perm[i:i + CFG['m2_bs']]
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                loss = trades_loss(net, IMGS[IMGIDX[b]], Q0[b], Y[b], beta, eps, stp, steps, CFG['trades_noise'])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        vc, vr = val_metrics(net); score = 0.5 * (vc + vr)
        ck = dict(visual=net.visual.state_dict(), head=net.head.state_dict(), epoch=ep, seed=seed, beta=beta, train_eps=CFG['trades_eps'],
                  train_steps=steps, val_clean=vc, val_ra=vr, cfg_hash=CFG_HASH)
        if CFG['m2_save_all_epochs']: torch.save(ck, f'{CK}/{name}_ep{ep}.pt')
        if score > best: best = score; torch.save(ck, f'{CK}/{name}_bestsofar.pt')
        torch.save(dict(visual=net.visual.state_dict(), head=net.head.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=ep, best=best, beta=beta, seed=seed), rp)
        print(f'[{name}] ep {ep} loss {loss.item():.3f} val_clean {vc:.3f} val_RA {vr:.3f} ({time.time()-t0:.0f}s)')
    shutil.copy(f'{CK}/{name}_bestsofar.pt', best_p); return best_p

def select_beta():
    sp = f'{ROOT}/beta_selection.json'
    if CFG['beta'] is not None: return CFG['beta']
    if os.path.exists(sp): return json.load(open(sp))['beta']
    s0 = CFG['seeds'][0]; rows = []
    for b in CFG['beta_grid']:
        p = train_model2(s0, b, CFG['beta_sel_epochs'], f'sel_beta{b}_seed{s0}', base_seed=s0)
        ck = torch.load(p, map_location='cpu'); rows.append(dict(beta=b, val_clean=ck['val_clean'], val_ra=ck['val_ra']))
    base_clean = max(r['val_clean'] for r in rows)
    ok = [r for r in rows if r['val_clean'] >= base_clean - CFG['max_clean_drop']] or rows
    best = max(ok, key=lambda r: r['val_ra'])
    json.dump(dict(beta=best['beta'], table=rows, rule='maks val_RA dengan val_clean tidak turun > max_clean_drop dari yang terbaik'), open(sp, 'w'), indent=1)
    print(pd.DataFrame(rows)); print('beta terpilih (validasi):', best['beta']); return best['beta']

BETA = select_beta()
for s in CFG['seeds']:
    train_model2(s, BETA, CFG['m2_epochs'], f'model2_seed{s}')
    if CFG['run_control']: train_model2(s, 0, CFG['m2_epochs'], f'model2ctl_seed{s}')

In [ ]:
# Skenario 5: ulangi skenario 1 sampai 4 pada Model 2, lalu sanity
run_scenarios('M2'); eval_open('M2'); apply_sanity('M2')
if CFG['run_control']:
    run_scenarios('M2ctl'); eval_open('M2ctl'); apply_sanity('M2ctl')

In [ ]:
# ============================ PEMERIKSAAN GRADIENT MASKING (FR-12) ============================
def gm_report(tag):
    recs = load_all(); tol = CFG['gm_tol']; rows = []
    clean = mean_acc(recs, tag, 'Clean', 0)
    def add(chk, det, val, ok): rows.append(dict(model=tag, pemeriksaan=chk, detail=det, nilai=val, lulus='LULUS' if ok else 'GAGAL'))
    # (a) black-box (Square) tidak boleh mengungguli white-box (PGD/AutoAttack)
    for e in CFG['eps_list']:
        sq, pg, aa = mean_acc(recs, tag, 'Square', e), mean_acc(recs, tag, 'PGD', e), mean_acc(recs, tag, 'AutoAttack', e)
        wb = [v for v in (pg, aa) if v is not None]
        if sq is not None and wb: add('(a) Square vs white-box', f'ε={e:g}/255', f'RA Square {sq:.3f} vs white-box terkuat {min(wb):.3f}', sq >= min(wb) - tol)
    # (b) RA tidak naik saat ε naik dan tidak melebihi akurasi bersih
    for atk in ['FGSM', 'PGD', 'AutoAttack']:
        es = sorted(e for (t, a, e, tx) in recs if t == tag and a == atk and tx == 'none' and e <= 8)
        ras = [mean_acc(recs, tag, atk, e) for e in es]
        mono = all(ras[i + 1] <= ras[i] + tol for i in range(len(ras) - 1)) if ras else True
        add('(b) monoton & <= bersih', atk, ' '.join(f'{e:g}:{r:.3f}' for e, r in zip(es, ras)), mono and (clean is None or all(r <= clean + tol for r in ras)))
    # (c) ε jauh lebih besar harus membuat akurasi mendekati nol
    big = mean_acc(recs, tag, 'PGD', CFG['gm_eps_large'])
    if big is not None: add('(c) ε besar', f'PGD ε={CFG["gm_eps_large"]}/255', f'RA {big:.3f}', big <= CFG['gm_zero_thr'])
    # (d) menggandakan iterasi PGD tidak boleh menaikkan keberhasilan serangan
    for e in CFG['gm_x2_eps']:
        a1, a2 = recs.get((tag, 'PGD', float(e), 'none')), recs.get((tag, 'PGDx2', float(e), 'none'))
        if a1 and a2:
            def asr(r): return np.mean([np.sum((v[1]['pred_orig'] == v[1]['y']) & (v[1]['pred_adv'] != v[1]['y'])) / max(1, np.sum(v[1]['pred_orig'] == v[1]['y'])) for v in r.values()])
            add('(d) iterasi x2', f'ε={e:g}/255', f'ASR {asr(a1):.3f} -> {asr(a2):.3f}', asr(a2) - asr(a1) <= tol)
    return pd.DataFrame(rows)

GM = pd.concat([gm_report(t) for t in ['M1', 'M2'] + (['M2ctl'] if CFG['run_control'] else [])], ignore_index=True)
GM.to_csv(f'{ROOT}/gradient_masking_report.csv', index=False)
print(GM.to_string(index=False))
print('\nBila ada GAGAL pada Model 2: curigai gradient masking, kuatkan serangan (lebih banyak langkah/restart, serangan adaptif BPDA/EOT bila ada preprocessing non-diferensiabel; FR-13), jangan klaim pertahanan berhasil.')

In [ ]:
# ============================ CP4 ============================
recs = load_all()
checkpoint('CP4 (TRADES + gradient masking). BATAS AKHIR EKSPERIMEN (minggu 11)', [
    ('Model 2 terlatih untuk semua seed', all(os.path.exists(f'{CK}/model2_seed{s}_best.pt') for s in CFG['seeds'])),
    ('skenario 5 selesai: Model 2 punya semua serangan utama', all(len(recs.get(('M2', a, float(e), 'none'), {})) == len(CFG['seeds']) for a in ['FGSM', 'PGD', 'AutoAttack'] for e in CFG['eps_list'])),
    ('empat pemeriksaan FR-12 lulus untuk Model 2', bool((GM[GM.model == 'M2'].lulus == 'LULUS').all()) and len(GM[GM.model == 'M2']) > 0),
], hard=False)

## 7. Hasil: tabel dan gambar (FR-14, FR-15, FR-16)

Semua angka dibaca dari `results/*.json`. **Jangan mengetik angka ke naskah secara manual** (CP5): salin dari CSV di `results/tables/`.
Interval kepercayaan 95% = bootstrap persentil pada **tingkat citra** (1.000 resampel), dengan metrik dirata-ratakan dari 3 seed. Pertanyaan pada citra yang sama dianggap saling bergantung (NFR-04).

In [ ]:
TAB = f'{RES}/tables'; os.makedirs(TAB, exist_ok=True)

def boot(fn, imgs, B=None, seed=0):
    # fn(w) -> dict nama->nilai, w = bobot sampel (jumlah kemunculan citra pada resampel)
    B = B or CFG['boot_B']
    uniq, inv = np.unique(imgs, return_inverse=True); U = len(uniq); rng = np.random.default_rng(seed)
    pt = fn(np.ones(len(imgs))); acc = {k: [] for k in pt}
    for _ in range(B):
        c = np.bincount(rng.integers(0, U, U), minlength=U)[inv].astype(float)
        try:
            v = fn(c)
            for k in pt: acc[k].append(v[k])
        except Exception: pass
    return {k: (pt[k], *np.nanpercentile(acc[k], [2.5, 97.5])) for k in pt}

def fmt(t, d=1): return f'{100*t[0]:.{d}f} [{100*t[1]:.{d}f}, {100*t[2]:.{d}f}]'

def R_of(recs, tag, atk, eps, text='none'):
    r = recs.get((tag, atk, float(eps), text))
    return None if not r else [r[s][1] for s in sorted(r)]

def f_acc(R, w): return float(np.mean([np.sum(w * (r['pred_adv'] == r['y'])) / np.sum(w) for r in R]))
def f_asr(R, w):
    v = []
    for r in R:
        ok = r['pred_orig'] == r['y']; v.append(np.sum(w * (ok & (r['pred_adv'] != r['y']))) / max(1e-9, np.sum(w * ok)))
    return float(np.mean(v))

def f_full(R, w):
    from sklearn.metrics import roc_auc_score
    keys = ['accuracy', 'balanced_acc', 'precision', 'recall', 'f1', 'auc', 'mcc', 'fnr']; out = {k: [] for k in keys}
    for r in R:
        y, p, s = r['y'], r['pred_adv'], r['p_adv']
        tp, tn = np.sum(w * ((y == 1) & (p == 1))), np.sum(w * ((y == 0) & (p == 0)))
        fp, fn = np.sum(w * ((y == 0) & (p == 1))), np.sum(w * ((y == 1) & (p == 0)))
        tpr, tnr = tp / (tp + fn + 1e-12), tn / (tn + fp + 1e-12); prec = tp / (tp + fp + 1e-12)
        out['accuracy'].append((tp + tn) / w.sum()); out['balanced_acc'].append((tpr + tnr) / 2); out['precision'].append(prec)
        out['recall'].append(tpr); out['f1'].append(2 * prec * tpr / (prec + tpr + 1e-12))
        out['mcc'].append((tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn) + 1e-12))
        out['fnr'].append(fn / (tp + fn + 1e-12))
        out['auc'].append(roc_auc_score(y, s, sample_weight=w) if (w[y == 1].sum() > 0 and w[y == 0].sum() > 0) else np.nan)
    return {k: float(np.nanmean(v)) for k, v in out.items()}

recs = load_all()

# ---- Tabel 1: skenario 1, data bersih (G1, FR-14)
rows = []
for tag in ['M1', 'M2'] + (['M2ctl'] if CFG['run_control'] else []):
    R = R_of(recs, tag, 'Clean', 0)
    if R is None: continue
    b = boot(lambda w: f_full(R, w), R[0]['img'])
    rows.append(dict(model=tag, **{k: fmt(v, 1) if k != 'mcc' else f'{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]' for k, v in b.items()}))
T1 = pd.DataFrame(rows); T1.to_csv(f'{TAB}/T1_clean.csv', index=False); print('Tabel 1. Data bersih, ya/tidak, rata-rata 3 seed [CI 95% bootstrap tingkat citra] (%)'); display(T1)

# ---- Confusion matrix (jumlah dari semua seed) dan sensitivitas per kelas
for tag in ['M1', 'M2']:
    R = R_of(recs, tag, 'Clean', 0)
    if R is None: continue
    cm = np.zeros((2, 2), int)
    for r in R:
        for t in (0, 1):
            for p in (0, 1): cm[t, p] += int(np.sum((r['y'] == t) & (r['pred_adv'] == p)))
    print(f'Confusion matrix {tag} (baris=benar, kolom=prediksi, urutan [no, yes], dijumlah {len(R)} seed)'); display(pd.DataFrame(cm, index=['benar no', 'benar yes'], columns=['pred no', 'pred yes']))

# ---- Tabel 2: template hasil utama, robust accuracy (G2)
eps_all = sorted(set(CFG['eps_list']) | {e for e in CFG['eps_extra'] if ('M1', 'PGD', float(e), 'none') in recs})
rows = []; rows_asr = []
for tag in ['M1', 'M2'] + (['M2ctl'] if CFG['run_control'] else []):
    for atk in ['FGSM', 'PGD', 'AutoAttack']:
        r1, r2 = dict(Model=tag, Serangan=atk), dict(Model=tag, Serangan=atk)
        for e in eps_all:
            R = R_of(recs, tag, atk, e)
            if R is None: r1[f'ε={e:g}/255'] = r2[f'ε={e:g}/255'] = ''; continue
            n = len(R[0]['y']); star = '*' if (atk == 'AutoAttack' and CFG['aa_n_samples'] and n < len(idx_test_cl)) else ''
            b = boot(lambda w: dict(ra=f_acc(R, w), asr=f_asr(R, w)), R[0]['img'])
            r1[f'ε={e:g}/255'] = fmt(b['ra']) + star; r2[f'ε={e:g}/255'] = fmt(b['asr']) + star
        rows.append(r1); rows_asr.append(r2)
T2 = pd.DataFrame(rows); T2.to_csv(f'{TAB}/T2_robust_accuracy.csv', index=False)
T2b = pd.DataFrame(rows_asr); T2b.to_csv(f'{TAB}/T2b_asr.csv', index=False)
print('Tabel 2. Robust accuracy RA(ε) (%), rata-rata 3 seed [CI 95%]   (* = AutoAttack pada subset)'); display(T2)
print('Tabel 2b. Attack success rate ASR(ε) (%) = bagian yang awalnya benar lalu salah'); display(T2b)

# ---- konfigurasi PGD ala SafeMed-R1 (FR-09)
rows = []
for tag in ['M1', 'M2']:
    ks = [k for k in recs if k[0] == tag and k[1] == 'PGDsafemed']
    for k in ks:
        R = [v[1] for v in recs[k].values()]; b = boot(lambda w: dict(ra=f_acc(R, w)), R[0]['img'])
        rows.append(dict(Model=tag, Konfigurasi='PGD ε=0,01 (≈2,55/255), α=0,002, 5 langkah', RA=fmt(b['ra'])))
T2c = pd.DataFrame(rows); T2c.to_csv(f'{TAB}/T2c_safemed_config.csv', index=False); display(T2c)
print('CATATAN (NFR-06): bandingkan dengan angka SafeMed-R1 hanya secara KUALITATIF. Dataset, model, dan parameter evaluasi berbeda (SafeMed-R1 tidak menyatakan parameter evaluasinya).')

In [ ]:
# ---- Tabel 3 dan 4: perturbasi teks (UJI ROBUSTNESS) dan gabungan (G3)
rows = []
for tag in ['M1', 'M2']:
    Rc = R_of(recs, tag, 'Clean', 0)
    if Rc is None: continue
    for t in CFG['text_kinds']:
        R = R_of(recs, tag, 'Clean', 0, t)
        if R is None: continue
        b = boot(lambda w: dict(acc=f_acc(R, w), drop=f_acc(Rc, w) - f_acc(R, w)), R[0]['img'])
        rows.append(dict(Model=tag, Perubahan_teks=t + ' (uji robustness)', Akurasi=fmt(b['acc']), Penurunan_poin=fmt(b['drop'])))
T3 = pd.DataFrame(rows); T3.to_csv(f'{TAB}/T3_text.csv', index=False); print('Tabel 3. Perturbasi teks saja'); display(T3)

rows = []
for tag in ['M1', 'M2']:
    for t in CFG['text_kinds']:
        for atk in CFG['combo_attacks']:
            for e in CFG['combo_eps']:
                R = R_of(recs, tag, atk, e, t); Ri = R_of(recs, tag, atk, e)
                if R is None or Ri is None: continue
                b = boot(lambda w: dict(ra=f_acc(R, w), asr=f_asr(R, w), ra_img=f_acc(Ri, w)), R[0]['img'])
                rows.append(dict(Model=tag, Serangan=f'{atk} ε={e:g}/255', Teks=t, RA_gabungan=fmt(b['ra']), RA_citra_saja=fmt(b['ra_img']), ASR_gabungan=fmt(b['asr'])))
T4 = pd.DataFrame(rows); T4.to_csv(f'{TAB}/T4_combined.csv', index=False); print('Tabel 4. Serangan citra + perubahan teks'); display(T4)

# ---- pertanyaan terbuka (FR-16), dilaporkan terpisah
rows = []
for tag in ['M1', 'M2'] + (['M2ctl'] if CFG['run_control'] else []):
    fs = sorted(glob.glob(f'{RES}/OPEN_{tag}_seed*.json'))
    if not fs: continue
    D = [json.load(open(f)) for f in fs]; img = np.array(D[0]['img']); C = [np.array(d['correct']) for d in D]
    b = boot(lambda w: dict(em=float(np.mean([np.sum(w * c) / w.sum() for c in C]))), img)
    rows.append(dict(Model=tag, Exact_match_terbuka=fmt(b['em']), n=len(img)))
T5 = pd.DataFrame(rows); T5.to_csv(f'{TAB}/T5_open.csv', index=False); print('Tabel 5. Pertanyaan terbuka, exact match, data bersih (dilaporkan terpisah)'); display(T5)

# ---- K1 dan K2 (usulan, belum disepakati)
rows = []
for atk in ['PGD', 'AutoAttack']:
    for e in [4, 8]:
        R1, R2 = R_of(recs, 'M1', atk, e), R_of(recs, 'M2', atk, e)
        if R1 is None or R2 is None: continue
        b = boot(lambda w: dict(d=f_acc(R2, w) - f_acc(R1, w)), R1[0]['img'])['d']
        rows.append(dict(kriteria='K1', cek=f'RA Model 2 - Model 1, {atk} ε={e}/255 (poin)', nilai=fmt(b), lulus='LULUS' if b[1] > 0 else 'belum terbukti (CI mencakup 0)'))
R1, R2 = R_of(recs, 'M1', 'Clean', 0), R_of(recs, 'M2', 'Clean', 0)
if R1 and R2:
    b = boot(lambda w: dict(d=f_acc(R1, w) - f_acc(R2, w)), R1[0]['img'])['d']
    rows.append(dict(kriteria='K2', cek='Penurunan akurasi bersih Model 1 - Model 2 (poin)', nilai=fmt(b), lulus='LULUS' if b[0] < CFG['max_clean_drop'] else 'GAGAL'))
K = pd.DataFrame(rows); K.to_csv(f'{TAB}/K1_K2.csv', index=False); display(K)
print('Catatan desain: Model 2 membuka encoder SEKALIGUS memakai TRADES. Kenaikan robustness belum tentu berasal dari TRADES saja. ' + ('Lihat model kontrol M2ctl.' if CFG['run_control'] else 'Aktifkan CFG["run_control"] untuk memisahkan efeknya.'))

In [ ]:
# ---- Gambar: kurva robust accuracy terhadap ε (G2)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = {'M1': '#c0392b', 'M2': '#1f77b4', 'M2ctl': '#7f8c8d'}
for ax, atk in zip(axes, ['FGSM', 'PGD', 'AutoAttack']):
    for tag in ['M1', 'M2'] + (['M2ctl'] if CFG['run_control'] else []):
        xs, m, lo, hi = [], [], [], []
        R0 = R_of(recs, tag, 'Clean', 0)
        if R0 is not None:
            b = boot(lambda w: dict(a=f_acc(R0, w)), R0[0]['img'])['a']; xs.append(0); m.append(b[0]); lo.append(b[1]); hi.append(b[2])
        for e in eps_all:
            R = R_of(recs, tag, atk, e)
            if R is None: continue
            b = boot(lambda w: dict(a=f_acc(R, w)), R[0]['img'])['a']; xs.append(e); m.append(b[0]); lo.append(b[1]); hi.append(b[2])
        if xs: ax.plot(xs, m, marker='o', color=colors[tag], label=tag); ax.fill_between(xs, lo, hi, color=colors[tag], alpha=.15)
    ax.set_title(atk); ax.set_xlabel('ε (per 255)'); ax.set_ylim(0, 1); ax.grid(alpha=.3)
axes[0].set_ylabel('Robust accuracy (ya/tidak)'); axes[0].legend()
plt.suptitle('Robust accuracy vs ε, rata-rata 3 seed, pita = CI 95% bootstrap (tingkat citra)')
plt.tight_layout(); plt.savefig(f'{FIG}/ra_vs_eps.png', dpi=150); plt.show()

In [ ]:
# ============================ CP5: bekukan hasil sebelum menulis ============================
manifest = {os.path.basename(p): hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16] for p in sorted(glob.glob(f'{RES}/*.json')) + sorted(glob.glob(f'{TAB}/*.csv'))}
json.dump(manifest, open(f'{ROOT}/results_manifest_{datetime.date.today()}.json', 'w'), indent=1)
print(len(manifest), 'berkas hasil dibekukan (hash tercatat). Salin angka naskah dari', TAB, 'dan jangan ketik manual.')
print('Hasil yang gagal sanity ada di', REJ, ':', len(glob.glob(f'{REJ}/*.json')) - (1 if os.path.exists(f"{REJ}/rejected_log.json") else 0), 'berkas (jelaskan di keterbatasan).')

## 8. Demo Gradio (FR-17, FR-18, R-13)

Demo hanya **memuat checkpoint final** (tidak melatih atau mengubah model) dan selalu menampilkan disclaimer. Pilih contoh dari set uji atau unggah citra, ajukan pertanyaan, atur ε, lalu lihat jawaban, confidence, dan perubahan setelah serangan.
Pada demo tidak ada label benar, jadi serangan memakai jawaban bersih model sebagai target yang dijauhkan. Demo memakai kosakata jawaban penuh (bukan 2 logit) dan seed pertama.

In [ ]:
import gradio as gr
DISCLAIMER = '## ⚠️ BUKAN ALAT DIAGNOSIS\nDemo ini hanya untuk penelitian dan pembelajaran tentang ketahanan model. **Jangan dipakai untuk keputusan klinis.**'
_DN = {}
def demo_net(name):
    tag = 'M1' if name.startswith('Model 1') else 'M2'
    if tag not in _DN: _DN[tag] = get_net(tag, CFG['seeds'][0])      # hanya memuat checkpoint final (R-13)
    return _DN[tag]

class WrapFull(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net; self.q = None
    def forward(self, x): return self.net(x, self.q)

def to_pil(t): return TF.to_pil_image(t.detach().clamp(0, 1).cpu())
SAMPLES = {f'#{int(i)} | {df.question.values[int(i)]}': int(i) for i in idx_test_cl[:60].cpu().numpy()}

def demo_run(model_name, upload, sample, question, attack, eps):
    if upload is not None:
        x = TF.to_tensor(upload.convert('RGB').resize((S, S), Image.BICUBIC)).unsqueeze(0).to(DEV)
        question = question or 'Is there an abnormality?'
    else:
        i = SAMPLES[sample]; x = IMGS[IMGIDX[i]].unsqueeze(0); question = question or df.question.values[i]
    net = demo_net(model_name); wrap = WrapFull(net).eval(); wrap.q = embed_text([question])
    with torch.no_grad(): p0 = F.softmax(wrap(x), 1)
    c0 = int(p0.argmax(1)); xa = x
    if eps > 0:
        atk = make_attack(wrap, attack, float(eps), 0); xa = atk(x, torch.tensor([c0], device=DEV))
    with torch.no_grad(): p1 = F.softmax(wrap(xa), 1)
    c1 = int(p1.argmax(1))
    txt = (f'**Pertanyaan:** {question}\n\n**Sebelum serangan:** {VOCAB[c0]} (confidence {float(p0.max()):.1%})\n\n'
           f'**Sesudah serangan ({attack}, ε={eps}/255):** {VOCAB[c1]} (confidence {float(p1.max()):.1%})\n\n'
           + ('**Jawaban BERUBAH.**' if c0 != c1 else 'Jawaban tidak berubah.') + '\n\n---\n⚠️ Bukan alat diagnosis. Tidak untuk keputusan klinis.')
    return to_pil(x[0]), to_pil(xa[0]), txt

with gr.Blocks(title='Demo Ketahanan VQA Medis') as demo:
    gr.Markdown(DISCLAIMER)
    with gr.Row():
        with gr.Column():
            m = gr.Radio(['Model 1 (BiomedCLIP beku)', 'Model 2 (TRADES)'], value='Model 1 (BiomedCLIP beku)', label='Model')
            up = gr.Image(type='pil', label='Unggah citra (opsional)'); sm = gr.Dropdown(list(SAMPLES), value=list(SAMPLES)[0], label='Atau contoh dari set uji')
            q = gr.Textbox(label='Pertanyaan (kosong = pakai pertanyaan contoh)')
            a = gr.Radio(['FGSM', 'PGD'], value='PGD', label='Serangan'); e = gr.Slider(0, 8, value=4, step=0.5, label='ε (per 255)')
            go = gr.Button('Jalankan')
        with gr.Column():
            i1 = gr.Image(label='Citra asli'); i2 = gr.Image(label='Citra setelah serangan'); out = gr.Markdown()
    go.click(demo_run, [m, up, sm, q, a, e], [i1, i2, out])
    gr.Markdown(DISCLAIMER)
if CFG['launch_demo']:
    demo.launch(share=True, debug=False)

In [ ]:
# ============================ CP6 ============================
checkpoint('CP6 (demo)', [
    ('disclaimer "bukan alat diagnosis" ada di UI', 'BUKAN ALAT DIAGNOSIS' in DISCLAIMER),
    ('demo memuat checkpoint final, tidak melatih (R-13)', os.path.exists(f'{CK}/model1_seed{CFG["seeds"][0]}_best.pt')),
], hard=False)

## Catatan keterbatasan (isi saat menulis laporan)

- Set uji resmi kecil (451 pasangan, sekitar separuhnya ya/tidak): satu pertanyaan sekitar 0,22 poin. Selalu baca interval kepercayaan, jangan hanya rata-rata.
- Perbandingan dengan SafeMed-R1 hanya kualitatif (NFR-06).
- Model 2 mengubah dua hal sekaligus (encoder dibuka dan TRADES); lihat model kontrol.
- Perturbasi teks adalah uji robustness, bukan serangan adversarial.
- Hasil yang gagal sanity ada di `results_rejected/` beserta alasannya.